# Coevolution with Bayesian Updates

In [71]:
# CONSTANTS: defining constants for evolutionary algorithm parameters

INITIAL_CODE_POPULATION_SIZE = 10
INITIAL_TEST_POPULATION_SIZE = 20

C_0 = 0.4 # Prior probability of a code being correct
T_0 = 0.6 # Prior probability of a test being correct


# Simulation constants
PROBABILITY_OF_CODE_TEST_PASSING = 0.7

In [72]:
import numpy as np

C: np.ndarray = np.array([C_0 for _ in range(INITIAL_CODE_POPULATION_SIZE)])
T: np.ndarray = np.array([T_0 for _ in range(INITIAL_TEST_POPULATION_SIZE)])

print("Initial Code Population Probabilities:", C)
print("Initial Test Population Probabilities:", T)

Initial Code Population Probabilities: [0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4 0.4]
Initial Test Population Probabilities: [0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6 0.6
 0.6 0.6]


# Initial Code-Test Evaluation

In [73]:
def _evaluate_code_test_pair(code_idx: int, test_idx: int) -> int:
    # Simulation of code-test evaluation
    # Randomly return True (pass) or False (fail)
    return int(np.random.rand() < PROBABILITY_OF_CODE_TEST_PASSING)

def evaluate_population(C: np.ndarray, T: np.ndarray) -> np.ndarray:
    matrix = np.zeros((C.shape[0], T.shape[0]), dtype=np.int8)
    for i in range(C.shape[0]):
        for j in range(T.shape[0]):
            matrix[i, j] = _evaluate_code_test_pair(i, j)
    return matrix

matrix = evaluate_population(C, T)
matrix  # Convert boolean to int for better visualization

array([[1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1],
       [0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1],
       [1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1],
       [1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1],
       [1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0],
       [0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0],
       [1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0],
       [1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1],
       [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0]],
      dtype=int8)

# Posterior Probability Update

In [74]:
def _get_posterior_update(X: float, Y: float, observation: bool) -> float:
    """
    Computes the updated belief for a code or test.
    Args:
        X: Previous belief that we will be updating
        Y: Previous belief that determines the update (e.g., if updating code belief, Y is test belief)
        observation: Result of the test case (True for pass, False for fail)
    """

    if observation:
        k = (1 - Y) / Y
    else:
        k = Y / (1 - Y)

    ratio = (k) * ((1 - X) / X)
    return 1 / (1 + ratio)


def get_posterior_update_for_code(code_prob: float, test_prob:float, observation: bool) -> float:
    return _get_posterior_update(code_prob, test_prob, observation)

def get_posterior_update_for_test(test_prob: float, code_prob:float, observation: bool) -> float:
    return _get_posterior_update(test_prob, code_prob, observation)


def update_all_probabilities(C_prev: np.ndarray, T_prev: np.ndarray, matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Demonstrates the Bayesian update for all codes and tests based on the evaluation matrix.
    Args:
        C: Array of probabilities that each code is correct
        T: Array of probabilities that each test is correct
        matrix: Evaluation matrix where each entry is 1 (pass) or 0 (fail)
    """

    C_new = np.zeros_like(C_prev)
    T_new = np.zeros_like(T_prev)


    print("=== Posterior Update ===")
    print("--- Updating Test Probabilities ---")
    for j in range(T_prev.shape[0]):
        print("Previous Test Probability:", T_prev[j])

        test_prob = T_prev[j]
        for i in range(C_prev.shape[0]):
            code_prob = C_prev[i]
            observation = bool(matrix[i, j])

            test_prob = get_posterior_update_for_test(test_prob, code_prob, observation)
            print(f"  Code {i} (Prob: {code_prob}, Observation: {observation}) -> Updated Test Prob: {test_prob}")
        
        T_new[j] = test_prob
        print("Updated Test Probability:", T_new[j])


    print("--- Updating Code Probabilities ---")
    for i in range(C_prev.shape[0]):
        print("Previous Code Probability:", C_prev[i])

        code_prob = C_prev[i]
        for j in range(T_prev.shape[0]):
            test_prob = T_prev[j]
            observation = bool(matrix[i, j])

            code_prob = get_posterior_update_for_code(code_prob, test_prob, observation)
            print(f"  Test {j} (Prob: {test_prob}, Observation: {observation}) -> Updated Code Prob: {code_prob}")
        
        C_new[i] = code_prob
        print("Updated Code Probability:", C_new[i])


    return C_new, T_new
    
for i in range(1):
    print(f"=== Iteration {i+1} ===")
    C, T = update_all_probabilities(C, T, matrix)
    print("Updated Code Population Probabilities:", C)
    print("Updated Test Population Probabilities:", T)

=== Iteration 1 ===
=== Posterior Update ===
--- Updating Test Probabilities ---
Previous Test Probability: 0.6
  Code 0 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.5
  Code 1 (Prob: 0.4, Observation: False) -> Updated Test Prob: 0.6
  Code 2 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.5
  Code 3 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.4
  Code 4 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.3076923076923077
  Code 5 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.22857142857142856
  Code 6 (Prob: 0.4, Observation: False) -> Updated Test Prob: 0.30769230769230765
  Code 7 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.22857142857142856
  Code 8 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.16494845360824742
  Code 9 (Prob: 0.4, Observation: True) -> Updated Test Prob: 0.11636363636363636
Updated Test Probability: 0.11636363636363636
Previous Test Probability: 0.6
  Code 0 (Prob: 0.4, Observation: True) -> Updated Tes